# Evaluation — Model Performance Analysis

This notebook evaluates the trained GRU and Transformer models on the held-out test set.

**Metrics:** Accuracy (Hit@1), Precision@K, Recall@K

**Prerequisites:** Run the Modeling notebook first to train and save model checkpoints to `models/`.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn

sys.path.insert(0, '..')
from src.models import GRURecommender, TransformerRecommender

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

PROCESSED = Path('../processed')
OUT       = Path('outputs'); OUT.mkdir(exist_ok=True)

_ts = datetime.now().strftime('%Y%m%d_%H%M')

# ── Load data ────────────────────────────────────────────────────────────────
vocab     = pd.read_parquet(PROCESSED / 'track_vocab.parquet')
test_seqs = pd.read_parquet(PROCESSED / 'test_seqs.parquet')
print(f'Vocab: {len(vocab):,} tracks')
print(f'Test:  {len(test_seqs):,} sequences')

# ── Parameters (must match training) ─────────────────────────────────────────
VOCAB_LIMIT = 100_000
MAX_SEQ_LEN = 50
EMBED_DIM   = 128
HIDDEN_DIM  = 256
NUM_LAYERS  = 2
NUM_HEADS   = 4
DROPOUT     = 0.2

PAD_IDX    = VOCAB_LIMIT
UNK_IDX    = VOCAB_LIMIT + 1
NUM_TOKENS = VOCAB_LIMIT + 2

SEED_RATIO = 0.8
K_VALUES   = [1, 5, 10, 20]

---
## 1. Load Trained Models

Instantiate both architectures (from `src/models.py`) and load the best checkpoints saved during training.

In [ ]:
gru_model = GRURecommender(
    NUM_TOKENS, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, PAD_IDX
).to(device)
gru_model.load_state_dict(torch.load('../models/gru_best.pt', map_location=device, weights_only=True))
gru_model.eval()
print(f'GRU loaded         ({sum(p.numel() for p in gru_model.parameters()):,} params)')

transformer_model = TransformerRecommender(
    NUM_TOKENS, EMBED_DIM, NUM_HEADS, NUM_LAYERS, DROPOUT, MAX_SEQ_LEN, PAD_IDX
).to(device)
transformer_model.load_state_dict(torch.load('../models/transformer_best.pt', map_location=device, weights_only=True))
transformer_model.eval()
print(f'Transformer loaded  ({sum(p.numel() for p in transformer_model.parameters()):,} params)')

---
## 2. Evaluation Metrics

We evaluate with the metrics specified in the assignment:
- **Accuracy (Hit@1)** — binary indicator: 1 if the single highest-ranked predicted track is exactly the true next track, 0 otherwise. Averaged over all test playlists.
- **Precision@K** — `|top-K ∩ holdout| / K`: of the K highest-ranked predicted tracks, how many appear in the hold-out continuation? Ranges from 0 to 1; penalizes irrelevant predictions.
- **Recall@K** — `|top-K ∩ holdout| / |holdout|`: of all tracks in the held-out continuation, how many are covered by the top-K predictions? Ranges from 0 to 1; penalizes missing relevant tracks.

**Protocol:** Each test playlist is split into a **seed** (first 80% of tracks) and a **holdout** (remaining 20%). The model receives the seed and its top-K logits from the last seed position are ranked to produce candidates, which are then compared against the holdout set.

In [ ]:
@torch.no_grad()
def evaluate_recsys(model, seqs_df, vocab_limit, max_len, dev,
                    seed_ratio=0.8, ks=[1, 5, 10, 20], max_eval=5000):
    """Recommendation evaluation: seed -> predict -> compare with holdout."""
    model.eval()
    unk = vocab_limit + 1
    metrics = {'Accuracy': []}
    for k in ks:
        metrics[f'Prec@{k}'] = []
        metrics[f'Recall@{k}'] = []

    seqs = seqs_df['track_idxs'].tolist()
    if len(seqs) > max_eval:
        seqs = [seqs[i] for i in np.random.default_rng(42).choice(len(seqs), max_eval, replace=False)]

    n_eval = 0
    for idxs in seqs:
        mapped = [i if i < vocab_limit else unk for i in idxs]
        sp = max(1, int(len(mapped) * seed_ratio))
        if sp >= len(mapped) - 1:
            continue
        seed = mapped[:sp]
        if len(seed) > max_len:
            seed = seed[-max_len:]
        holdout = set(mapped[sp:]) - {unk}
        if not holdout:
            continue

        inp   = torch.tensor([seed], dtype=torch.long, device=dev)
        pmask = torch.zeros(1, len(seed), dtype=torch.bool, device=dev)
        logits = model(inp, pad_mask=pmask)[0, -1, :vocab_limit]
        topk   = logits.topk(max(ks)).indices.tolist()

        metrics['Accuracy'].append(1 if topk[0] == mapped[sp] else 0)
        for k in ks:
            hits = len(set(topk[:k]) & holdout)
            metrics[f'Prec@{k}'].append(hits / k)
            metrics[f'Recall@{k}'].append(hits / len(holdout))
        n_eval += 1

    print(f'  Evaluated {n_eval:,} playlists')
    return {m: np.mean(v) for m, v in metrics.items()}

---
## 3. Results

In [ ]:
print('GRU:')
gru_metrics = evaluate_recsys(gru_model, test_seqs, VOCAB_LIMIT, MAX_SEQ_LEN, device, ks=K_VALUES)

print('Transformer:')
tf_metrics = evaluate_recsys(transformer_model, test_seqs, VOCAB_LIMIT, MAX_SEQ_LEN, device, ks=K_VALUES)

# add both metrics to result
results_df = pd.DataFrame({
    'Metric': list(gru_metrics.keys()),
    'GRU': list(gru_metrics.values()),
    'Transformer': list(tf_metrics.values())
})

### Note on Evaluation Sample Size

Out of 100,001 test sequences, only 4,756 were actually evaluated. This is because sequences where the seed covers the entire sequence (i.e. `sp >= len(mapped) - 1`) are skipped — there are no holdout tracks left to evaluate against. This primarily affects very short playlists. The evaluated subset is representative of playlists with at least a few tracks in the holdout.

### 3.1 Interpretation

The GRU outperforms the Transformer across every metric, which is somewhat counterintuitive given the Transformer's superior capacity for long-range dependencies. This is likely because the sequential patterns in playlists are predominantly local — the next track is most strongly predicted by the last few tracks, not by distant earlier ones. The GRU's inductive bias toward recent context therefore suits this task better than the Transformer's global attention.

Both models plateau early — validation accuracy for the GRU stabilises around epoch 9–10 and the Transformer around epoch 6–7, suggesting the models have extracted most of the learnable signal from the data at this point. Neither model shows signs of overfitting: validation loss continues to improve alongside training loss throughout training.

Accuracy `(Hit@1)` of `~6%` means the single highest-ranked track is the correct next track 1 in 16 times, predicting from a vocabulary of 100,000 candidates. `Recall@20` of `~12.7%` for the GRU means that given 20 recommendations, the model recovers roughly 1 in 8 of the user's actual future tracks — a more practically useful figure, since real recommenders surface lists rather than single items.

---
## 4. Summary & Best Model

In [ ]:
print('Final Model Comparison')
print('=' * 70)
summary = pd.DataFrame({'GRU': gru_metrics, 'Transformer': tf_metrics}).T
print(summary.to_string(float_format='%.4f'))

gru_r10 = gru_metrics.get('Recall@10', 0)
tf_r10  = tf_metrics.get('Recall@10', 0)
best_name = 'GRU' if gru_r10 >= tf_r10 else 'Transformer'
print(f'\nBest model by Recall@10: {best_name}')
print(f'Checkpoint: models/{best_name.lower()}_best.pt')

results_path = f'../models/results_summary_{_ts}.csv'
summary.to_csv(results_path)
print(f'Results saved to {results_path}')

## 5. Conclusions

1. **Sequential models learn playlist patterns** — both GRU and Transformer capture sequential structure in playlists, producing meaningful next-track recommendations from learned track embeddings trained entirely on co-occurrence data with no external features.
2. **GRU outperforms Transformer** on this task — local sequential context dominates over long-range dependencies in playlist continuation, favouring the GRU's recurrent inductive bias.
3. **Power-law popularity distribution** shapes performance — popular tracks are easier to predict, while the long tail of 2.26M unique tracks remains largely out of reach.
4. **Vocabulary truncation to 100k tracks** covers 84.2% of tokens but means 15.8% of sequence positions are collapsed to a single UNK token, capping the theoretical maximum recall.
5. **Neither model overfits** — validation accuracy tracks training accuracy throughout, suggesting more epochs or a larger model could still yield marginal gains, though the learning curves indicate diminishing returns beyond epoch 10.

## 6. Results & Comparison with State of the Art

### Our Results

| Model | Accuracy (Hit@1) | Prec@1 | Prec@5 | Prec@10 | Prec@20 | Recall@1 | Recall@5 | Recall@10 | Recall@20 |
|---|---|---|---|---|---|---|---|---|---|
| GRU | 0.0612 | 0.1178 | 0.0876 | 0.0720 | 0.0575 | 0.0157 | 0.0525 | 0.0828 | 0.1273 |
| Transformer | 0.0604 | 0.1074 | 0.0805 | 0.0681 | 0.0550 | 0.0142 | 0.0487 | 0.0786 | 0.1230 |

Both models were trained for 15 epochs on 800k sequences from the full MPD (1M playlists), with a vocabulary truncated to the top 100,000 tracks (84.2% coverage). The GRU outperforms the Transformer across all metrics and is selected as the primary model.

---

### Alternative Approach: Genre-Aware Popularity Recommender

As an alternative to the sequential models, we explored a genre-aware popularity recommender trained on the full dataset. The idea was to infer the genre of a playlist from its name using keyword matching, assign each track a genre by majority vote across the labeled playlists it appeared in, and recommend the most popular tracks from the inferred genre not already in the playlist.

| Model | Prec@1 | Prec@5 | Prec@10 | Prec@20 | Recall@1 | Recall@5 | Recall@10 | Recall@20 | Genre Accuracy |
|---|---|---|---|---|---|---|---|---|---|
| Genre Recommender | 0.0492 | 0.0438 | 0.0386 | 0.0322 | 0.0022 | 0.0105 | 0.0185 | 0.0309 | 0.6134 |

This approach did not perform well. The core problem is that **only ~40% of playlists have names that map to a recognizable genre** — the remaining 60% have generic names like "my playlist", "favorites", or "untitled" which carry no genre signal. Also we we relying on the fact that these genres we gathered from the playlist titles actually corresponded to the genre of the songs within them, which may not have been the case at all. This means the majority of the dataset provides no training signal for the genre inference step, fundamentally limiting coverage. Even among labeled playlists, many tracks span multiple genres (Drake appears in rap, party, workout, and pop playlists), making the majority-vote genre assignment noisy. The result is that the model is frequently recommending from the wrong genre entirely, which cascades into poor precision and recall regardless of how good the within-genre popularity ranking is. The sequential GRU and Transformer, which learn directly from track co-occurrence patterns across all playlists without relying on name heuristics, significantly outperform this approach across every metric.

---

### Comparison with ACM RecSys Challenge 2018

The task we solve is equivalent to the Automatic Playlist Continuation (APC) task defined in the ACM Recommender Systems Challenge 2018 (Zamani et al., 2019), which used the same Spotify Million Playlist Dataset. The challenge attracted 410 teams and 1,467 submissions. The best team achieved an R-precision of 0.2241 and NDCG of 0.3946.

Direct metric comparison is not straightforward because the challenge used different metrics (R-precision, NDCG, recommended song clicks) evaluated on 500 recommendations per playlist, while we use Precision@K and Recall@K on K ∈ {1, 5, 10, 20}. Our setup is closest to **Type 5** (first 10 tracks only, no title) or **Type 8** (first 100 tracks) from the challenge evaluation, depending on playlist length, as we provide the first 80% of tracks as seed with no title signal.

| Approach | Scale | R-precision / Prec@10 | Architecture |
|---|---|---|---|
| vl6 (1st place, ACM 2018) | 1M playlists | 0.2241 | WRMF + XGBoost (two-stage) |
| hello world! (2nd place, ACM 2018) | 1M playlists | 0.2234 | Autoencoder + CNN |
| Avito (3rd place, ACM 2018) | 1M playlists | 0.2153 | LightFM + XGBoost (two-stage) |
| **GRU (ours)** | **1M playlists** | **0.0720 (Prec@10)** | **Single GRU** |
| **Transformer (ours)** | **1M playlists** | **0.0681 (Prec@10)** | **Single Transformer** |

---

### Why Are Our Numbers Lower?

The gap between our results and the challenge winners is expected and can be attributed to several compounding factors:

**1. Model complexity and ensembling.** Every top-10 team in the challenge ensembled multiple models — matrix factorization, collaborative filtering, neural networks, and learning-to-rank re-rankers — in a two-stage architecture. Our models are single architectures with no ensembling or re-ranking stage. As noted in Zamani et al. (2019), the winning team (vl6) combined WRMF, item-item CF, user-user CF, a convolutional playlist embedder, and XGBoost into a unified pipeline. We train a single GRU or Transformer end-to-end.

**2. Evaluation metric differences.** The challenge's R-precision credits partial matches at the artist level (an artist match scores 0.25 even without an exact track match) and evaluates 500 recommendations per playlist. Our Precision@K and Recall@K only credit exact track matches and evaluate at K ≤ 20, making our metrics structurally stricter for equivalent model quality.

**3. Vocabulary truncation.** We truncate the track vocabulary to the top 100,000 tracks (84.2% token coverage), meaning 15.8% of track positions in sequences are mapped to a single UNK token. Any holdout track outside the top 100k is unrecoverable regardless of model quality. Challenge teams operated on the full 2.26M track vocabulary.

**4. Sequence length cap.** Our models receive at most the last 50 tracks of the seed as input due to the MAX_SEQ_LEN constraint. For long playlists this discards earlier context that could disambiguate the playlist's theme. The paper notes that models generally struggle with long sequences, but competitive teams used architectural solutions (autoencoders, attention) to mitigate this.

**5. No audio features.** Creative track teams used Spotify API audio features (danceability, energy, tempo, valence etc.) which encode musical content directly. We have no access to such features, limiting recommendations to co-occurrence patterns alone.

---

### Reference

Zamani, H., Schedl, M., Lamere, P., & Chen, C.W. (2019). *An Analysis of Approaches Taken in the ACM RecSys Challenge 2018 for Automatic Music Playlist Continuation.* ACM Transactions on Intelligent Systems and Technology. https://arxiv.org/abs/1810.01520